In [93]:
from clearml import Task, Dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
task = Task.init(
    project_name="Course Work",
    task_name="Data Preparation"
)

ClearML Task: created new task id=d862991be6cb4245ba4e59b19852fa09


Could not fetch GPU stats: NVML Shared Library Not Found


ClearML results page: https://app.clear.ml/projects/61047c7a66aa40eab8cd971dc2d8b828/experiments/d862991be6cb4245ba4e59b19852fa09/output/log


ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring
ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start


In [94]:
df = pd.read_csv("transaction.csv")
df["trDte"] = pd.to_datetime(df["trDte"], format="%d.%m.%Y")
df.head()

,trDte,bcode,clientID,item,itemGroup,quantity,amount
0,2017-09-01,code000000001,client13166,sku8444,Скобяные изделия,1,29
1,2017-09-01,code000000001,client13166,sku12545,Оборудование для сада и дачи,1,329
2,2017-09-01,code000000001,client13166,sku3391,Инструменты,1,169
3,2017-09-01,code000000001,client13166,sku20444,Инструменты,2,578
4,2017-09-01,code000000002,client1239,sku29959,Скобяные изделия,1,329


In [95]:
max_date = df["trDte"].max()
split_date = max_date - pd.Timedelta(days=30)

# Target = клиент сделал покупку после split_date
active_clients = df[df["trDte"] > split_date]["clientID"].unique()
rfm = pd.DataFrame({"clientID": df["clientID"].unique()})
rfm["target"] = rfm["clientID"].isin(active_clients).astype(int)

print(f"Баланс классов: {rfm['target'].mean():.3f}")

Баланс классов: 0.214


In [96]:
df_filtered = df[df["trDte"] <= split_date]  # только старые данные

# RFM-признаки
rfm_stats = df_filtered.groupby("clientID").agg({
    "trDte": lambda x: (split_date - x.max()).days,  # recency до split_date
    "clientID": "count",  # frequency
    "amount": ["sum", "mean"],  # monetary
    "quantity": ["sum", "mean"]
}).round(2)

rfm_stats.columns = ["recency", "frequency", "total_spent", "avg_check", "total_items", "avg_items"]
rfm = rfm.merge(rfm_stats, on="clientID", how="left")

# Признак выходных
df_filtered["weekday"] = df_filtered["trDte"].dt.weekday
df_filtered["is_weekend"] = df_filtered["weekday"].isin([5, 6]).astype(int)
weekend_stats = df_filtered.groupby("clientID")["is_weekend"].mean().reset_index()
weekend_stats.columns = ["clientID", "weekend_ratio"]

rfm = rfm.merge(weekend_stats, on="clientID", how="left")

rfm.fillna(0, inplace=True)  # если у клиента нет транзакций до split_date
rfm.head()

/var/folders/s2/rf8b2cln2wbgjl93s7fwjnp80000gn/T/ipykernel_57133/227570961.py:15: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/s2/rf8b2cln2wbgjl93s7fwjnp80000gn/T/ipykernel_57133/227570961.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,clientID,target,recency,frequency,total_spent,avg_check,total_items,avg_items,weekend_ratio
0,client13166,0,651.0,14.0,2705.0,193.21,23.0,1.64,0.000000
1,client1239,0,4.0,130.0,42161.0,324.32,219.0,1.68,0.353846
2,client30041,0,159.0,85.0,16057.0,188.91,133.0,1.56,0.129412
3,client36276,0,258.0,10.0,4614.0,461.40,12.0,1.20,0.100000
4,client14136,0,186.0,28.0,35870.0,1281.07,36.0,1.29,0.285714


In [97]:
train, test = train_test_split(
    rfm, test_size=0.2, random_state=42, stratify=rfm["target"]
)

train.to_csv("train.csv", index=False)
test.to_csv("test.csv", index=False)

In [98]:
train, test = train_test_split(
    rfm, test_size=0.2, random_state=42, stratify=rfm["target"]
)

train.to_csv("train.csv", index=False)
test.to_csv("test.csv", index=False)

In [99]:
dataset_raw = Dataset.create(
    dataset_name="Transactions Raw",
    dataset_project="Course Work"
)
dataset_raw.add_files("transaction.csv")
dataset_raw.upload()
dataset_raw.finalize()

dataset_features = Dataset.create(
    dataset_name="Transactions Features",
    dataset_project="Course Work"
)
dataset_features.add_files("train.csv")
dataset_features.add_files("test.csv")
dataset_features.upload()
dataset_features.finalize()

task.upload_artifact(name="train_csv", artifact_object="train.csv")
task.upload_artifact(name="test_csv", artifact_object="test.csv")

print("Подготовка данных завершена")
task.close()

ClearML results page: https://app.clear.ml/projects/e5583aeae2de439ba4a862792421735d/experiments/96b442a1239648b88e68d352f63fa3f4/output/log
ClearML dataset page: https://app.clear.ml/datasets/simple/e5583aeae2de439ba4a862792421735d/experiments/96b442a1239648b88e68d352f63fa3f4
Uploading dataset changes (1 files compressed to 9.89 MiB) to https://files.clear.ml
File compression and upload completed: total size 9.89 MiB, 1 chunk(s) stored (average size 9.89 MiB)
ClearML results page: https://app.clear.ml/projects/2851b3536d0a4697acf15f49e3a14fc8/experiments/1cc6aff4e1904719902ba8a23c898db4/output/log
ClearML dataset page: https://app.clear.ml/datasets/simple/2851b3536d0a4697acf15f49e3a14fc8/experiments/1cc6aff4e1904719902ba8a23c898db4
Uploading dataset changes (2 files compressed to 1.06 MiB) to https://files.clear.ml
File compression and upload completed: total size 1.06 MiB, 1 chunk(s) stored (average size 1.06 MiB)
Подготовка данных завершена
